# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`This notebook provides a guide for loading, exploring, and processing the FAIR² mental health survey dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd# Define the Croissant schema URLurl = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'# Load dataset metadatadataset = mlc.Dataset(url)metadata = dataset.metadata# Display metadata summaryprint(f"{metadata.name}: {metadata.description}")# Optional: Explore additional metadata fieldsprint(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")print(f"License: {getattr(metadata, 'license', 'N/A')}")print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data OverviewReview available record sets, fields, and their IDs.Entities in Croissant, such as record sets, fields, and columns, must be referenced by their `@id` values.

In [ ]:
# Examine available record sets and their IDsrecord_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []print('Available Record Sets:')# Each recordSet is referenced by '@id'for rs in record_sets:    print(f"  - @id: {getattr(rs, '@id', rs)}")# For this dataset, print details for the first record set (if present)if record_sets:    record_set_id = getattr(record_sets[0], '@id', record_sets[0])    # List fields in the record set, referenced by '@id'    fields = getattr(record_sets[0], 'field', [])    print(f"\nFields for record set @id '{record_set_id}':")    for f in fields:        print(f"    - @id: {getattr(f, '@id', f)} | name: {getattr(f, 'name', 'N/A')}")else:    print("No record sets found in metadata.")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis.Below, record sets and fields are referenced by their `@id` fields.

In [ ]:
# If record set IDs found, use them for extraction.record_set_ids = [getattr(rs, '@id', rs) for rs in record_sets]dataframes = {}for rs_obj in record_sets:    rs_id = getattr(rs_obj, '@id', rs_obj)    records = list(dataset.records(record_set=rs_id))    # Convert to DataFrame    df = pd.DataFrame(records)    dataframes[rs_id] = df    print(f"Loaded record set: {rs_id} | shape: {df.shape}")# Examine columns from first record setif record_set_ids:    print(f"\nData columns for record set '{record_set_ids[0]}':")    print(dataframes[record_set_ids[0]].columns.tolist())    # Show a preview    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.Entities must be referenced by their `@id` fields throughout.

In [ ]:
# Choose record set and fields for analysis.rs_id = record_set_ids[0] if record_set_ids else None# Inspect available columns to select numeric and grouping fields.df = dataframes[rs_id] if rs_id else pd.DataFrame()columns = df.columns.tolist()print(f"Columns in '{rs_id}':", columns)# Example: Select 'phq9_score' (by @id) as numeric field, 'village' as group field (or substitute based on available columns)numeric_field_id = 'phq9_score' if 'phq9_score' in columns else columns[0]group_field_id = 'village' if 'village' in columns else columns[1] if len(columns) > 1 else columns[0]# Filter out records with phq9_score > threshold (example threshold 10)threshold = 10if numeric_field_id in df.columns:    # Some columns may be string; ensure numeric    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')    filtered_df = df[df[numeric_field_id] > threshold].copy()    print(f"Filtered records with {numeric_field_id} > {threshold}:")    print(filtered_df.head())    # Normalize numeric field    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()    print(f"Normalized {numeric_field_id} for filtered records:")    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())    # Group by village (or selected group field)    if group_field_id in filtered_df.columns:        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")        print(grouped_df.head())else:    print(f"Numeric field '{numeric_field_id}' not found in dataframe.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.Here, we plot the distribution of PHQ-9 scores and group means by village.

In [ ]:
import matplotlib.pyplot as plt# Plot histogram of PHQ-9 score (numeric_field_id)if numeric_field_id in df.columns:    plt.figure(figsize=(7, 4))    df[numeric_field_id].dropna().hist(bins=20)    plt.title(f"Distribution of {numeric_field_id} (@id)")    plt.xlabel(numeric_field_id)    plt.ylabel("Frequency")    plt.show()# Plot grouped means by village (group_field_id)if 'grouped_df' in locals():    plt.figure(figsize=(10, 5))    plt.bar(grouped_df[group_field_id], grouped_df[f"mean_{numeric_field_id}"])    plt.title(f"Mean {numeric_field_id} by {group_field_id} (@id)")    plt.xlabel(group_field_id)    plt.ylabel(f"Mean {numeric_field_id}")    plt.xticks(rotation=45, ha='right')    plt.tight_layout()    plt.show()

## 6. ConclusionSummarize key findings and observations from the dataset exploration.- The dataset contains survey responses relevant to mental health from Kilifi County, Kenya, organized per Croissant record set and fields.- Using field `@id`s, we extracted and analyzed numeric scores (PHQ-9, GAD-7, PSQ) and demographic data.- Filtering by PHQ-9 score (`phq9_score` @id), we identified subsets of higher depressive symptoms and normalized score distributions.- Mean scores grouped by `village` (`village` @id) illustrate local variation in mental health indicators.This notebook illustrated FAIR data exploration workflows, referencing all dataset entities by their unique `@id` and leveraging Croissant and the `mlcroissant` library for reproducible analyses.